# Etapa 3 · Recuperación sobre el corpus jurídico

**Objetivo experimental:** comparar tres brazos de recuperación sobre el mismo corpus, preguntas congeladas y presupuesto de contexto: BM25, BGE-M3 denso, y fusión RRF. Se mantiene el benchmark de la Etapa 2 sin editarlo.

**Entradas:** `documents.jsonl` y `eval/FROZEN.json` con sus tres JSONL. **Salidas:** chunks, índice BM25, matriz de embeddings cacheada, métricas por ítem, intervalos bootstrap, comparaciones pareadas y una configuración exportable a las Etapas 4 y 5.

**Protocolo:** ejecutar en `dev` durante el desarrollo; reservar `test` para una única evaluación final una vez fijadas las decisiones. La revisión humana pendiente de la evaluación limita las conclusiones.

### 0. Preparación y recuperación de dependencias de Colab


In [40]:
# --- Bootstrap de Colab (no hace nada fuera de Colab) --------------------------------
# Colab arranca en /content con un entorno vacío: sin esta celda, el `import environment`
# siguiente falla con ModuleNotFoundError. La genera tools/py_to_notebook.py -- editala
# ahí, no acá.
import hashlib
import os
import subprocess
import sys

# El disco de Colab se borra cuando se desconecta el entorno de ejecución (~90 min inactivo,
# 12 h como máximo). Con USE_DRIVE = True, artifacts/ (documentos, embeddings, índices,
# resultados, cachés del LLM) vive en cambio en tu Google Drive, y al volver a correr se
# saltea todo lo que ya está construido.
USE_DRIVE = True
# Guardar también en Drive la caché de modelos de HuggingFace (~3 GB). Apagado por defecto:
# volver a descargar suele ser más rápido que leer archivos grandes a través del montaje de Drive.
MODELS_ON_DRIVE = True

# Un entorno de ejecución de Colab, desde el navegador o desde la extensión de Colab de VS Code
# (cuyo kernel puede no haber importado google.colab: la propia variable de entorno del runtime
# dice dónde estamos).
if "google.colab" in sys.modules or os.environ.get("COLAB_RELEASE_TAG"):
    REPO_URL = "https://github.com/Jahdejosefina/projecto_ragTP2.git"
    REPO_BRANCH = ""
    REPO_DIR = "projecto_rag"
    DRIVE = "/content/drive/MyDrive/projecto_rag"
    quiet = {**os.environ, "GIT_TERMINAL_PROMPT": "0"}  # nunca esperar un pedido de contraseña

    def github_token():
        # Un repositorio privado: un token de GitHub como secreto de Colab llamado GITHUB_TOKEN.
        # Los secretos de Colab solo se pueden leer desde el navegador; una variable de entorno
        # funciona en cualquier lado.
        token = os.environ.get("GITHUB_TOKEN")
        if not token:
            try:
                from google.colab import userdata
                token = userdata.get("GITHUB_TOKEN")
            except Exception:
                token = None
        return token

    # Dónde ESTAMOS antes que qué hay en disco: `isdir(REPO_DIR)` es relativo, así que
    # chequearlo primero clonaría una copia anidada cada vez que se vuelve a correr esta celda
    # en la misma sesión.
    if os.path.basename(os.getcwd()) != REPO_DIR:
        if not os.path.isdir(REPO_DIR):
            cmd = ["git", "clone", "--depth", "1"] + (["--branch", REPO_BRANCH] if REPO_BRANCH else [])
            if subprocess.run(cmd + [REPO_URL, REPO_DIR], env=quiet).returncode != 0:
                token = github_token()
                if not token:
                    raise SystemExit("git clone failed. Is the repository public? For a private one, "
                                     "add a Colab secret named GITHUB_TOKEN (in browser Colab: VS Code's "
                                     "extension can't read Colab secrets).")
                authed = REPO_URL.replace("https://", "https://x-access-token:" + token + "@")
                # capture_output: un mensaje de error no tiene que imprimir el token en el notebook
                if subprocess.run(cmd + [authed, REPO_DIR], env=quiet, capture_output=True).returncode != 0:
                    raise SystemExit("git clone failed even with GITHUB_TOKEN: check that the token "
                                     "can read this repository (fine-grained: Contents: Read).")
                # El token NO se guarda en .git/config: un pull posterior lo vuelve a pasar (abajo).
                subprocess.run(["git", "-C", REPO_DIR, "remote", "set-url", "origin", REPO_URL])
        os.chdir(REPO_DIR)
    # Traer los arreglos publicados después de tu primer clon (no hace nada si no hay).
    pulled = subprocess.run(["git", "pull", "--ff-only", "-q"], capture_output=True, text=True, env=quiet)
    token = github_token() if pulled.returncode != 0 else None
    if token:   # un repositorio privado: volver a hacer pull con el token, sin guardarlo
        branch = subprocess.run(["git", "rev-parse", "--abbrev-ref", "HEAD"], capture_output=True,
                                text=True).stdout.strip()
        authed = REPO_URL.replace("https://", "https://x-access-token:" + token + "@")
        pulled = subprocess.run(["git", "pull", "--ff-only", "-q", authed, branch], capture_output=True,
                                text=True, env=quiet)
    if pulled.returncode != 0:
        message = (pulled.stderr or pulled.stdout).strip()
        print("note: git pull skipped:", (message.replace(token, "***") if token else message)[:200])
    if os.getcwd() not in sys.path:
        sys.path.insert(0, os.getcwd())

    # Instalar una vez por VM: una marca con clave en el contenido de los archivos de requirements.
    reqs = [f for f in ("requirements.txt", "requirements-group.txt") if os.path.isfile(f)]
    digest = hashlib.sha256(b"".join(open(f, "rb").read() for f in reqs)).hexdigest()[:12]
    stamp = "/content/.lab_pip_" + digest
    if not os.path.exists(stamp):
        pip = [sys.executable, "-m", "pip", "install", "-q"]
        for f in reqs:
            pip += ["-r", f]
        subprocess.run(pip, check=True)  # UNA sola resolución: una versión del grupo que choca con una del núcleo falla ruidosamente
        open(stamp, "w").close()

    if MODELS_ON_DRIVE and not USE_DRIVE:
        print("note: MODELS_ON_DRIVE needs USE_DRIVE = True; the model cache stays on the VM disk.")
    if USE_DRIVE:
        # ANTES de `import environment`: un directorio artifacts/ real creado primero le ganaría
        # en silencio al link, y todo terminaría en el disco efímero.
        if not os.path.isdir("/content/drive/MyDrive"):   # VS Code: ya montado por su comando
            # Se avisa ANTES de la llamada: en la extensión de Colab de VS Code, drive.mount() puede colgarse en vez de fallar.
            print("Mounting Drive... (in VS Code, if this hangs: interrupt, run 'Colab: Mount Google Drive "
                  "to Server...' from Ctrl+Shift+P, then re-run this cell)")
            try:
                from google.colab import drive
                drive.mount("/content/drive")
            except Exception as e:  # drive.mount() desde una celda no funciona en la extensión de Colab de VS Code
                raise SystemExit("Drive is not mounted (" + type(e).__name__ + "). In VS Code: Ctrl+Shift+P > "
                                 "'Colab: Mount Google Drive to Server...', then re-run this cell.")
        os.makedirs(DRIVE + "/artifacts", exist_ok=True)
        if os.path.islink("artifacts"):
            print("artifacts ->", os.readlink("artifacts"))
        elif os.path.isdir("artifacts"):
            print("WARNING: artifacts/ is already a real directory, so it is NOT on Drive.")
            print("Move its contents into", DRIVE + "/artifacts", ", delete it, then re-run this cell.")
        else:
            os.symlink(DRIVE + "/artifacts", "artifacts")
            print("artifacts ->", DRIVE + "/artifacts")
        os.environ["LAB_USE_DRIVE"] = "1"
        if MODELS_ON_DRIVE:
            os.makedirs(DRIVE + "/hf_cache", exist_ok=True)
            os.environ.setdefault("HF_HOME", DRIVE + "/hf_cache")

    print("Colab bootstrap OK. Working directory:", os.getcwd())

artifacts -> /content/drive/MyDrive/projecto_rag/artifacts
Colab bootstrap OK. Working directory: /content/projecto_rag


### 0.1. Dependencias de recuperación densa

BGE-M3 requiere `sentence-transformers` y `torch`. Instalamos el paquete si falta; los pesos se descargan solamente cuando se construye el índice denso. La ejecución completa de BGE-M3 es recomendable en Colab con GPU T4.

In [41]:
import importlib.util, subprocess, sys
if importlib.util.find_spec("sentence_transformers") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "sentence-transformers>=3.0,<6"], check=True)

## 1. Entorno y decisiones fijadas antes de medir

Usamos 320 tokens por hijo, incluidos doc_id y título de sección, en el tokenizer del reranker.
El solapamiento objetivo es 15%; no se cruzan secciones ni se inventan padres.
El contexto entregado ocupa como máximo 2.500 tokens de Qwen3, incluidos IDs y separadores.
Este presupuesto es solo de pasajes: la etapa 4 deberá contar el prompt completo y reservar salida.

BM25 usa stemming español y la misma normalización léxica del diagnóstico de etapa 2.
El cambio de secciones completas a chunks impide interpretar el diagnóstico anterior como una fila pareada.
No se ajustan umbrales de abstención con los puntajes de las preguntas fuera del corpus.

In [42]:
import environment  # primero: mismas guardas del proyecto
from pathlib import Path
import importlib.util
import json
import random
import sys
import warnings

needed = [name for name in ['bm25s', 'Stemmer', 'transformers', 'huggingface_hub']
          if importlib.util.find_spec(name) is None]
if needed:
    raise RuntimeError(
        f'Faltan dependencias: {needed}. Instalá requirements-stage3.txt o ejecutá '
        '%pip install bm25s==0.2.14 PyStemmer==3.1.0 transformers==4.57.6 y reiniciá el kernel.'
    )

SMOKE_TEST = False
EVAL_SPLIT = 'test'
SEED = 1337
DOCUMENTS_PATH = environment.artifacts_dir(False) / 'documents.jsonl'
EVAL_DIR = None  # opcional: Path('/ruta/a/eval');
ART = environment.artifacts_dir(SMOKE_TEST)
PROFILE = 'smoke' if SMOKE_TEST else 'full'

if EVAL_SPLIT not in ('dev', 'test'):
    raise ValueError('Elegí dev o test explícitamente.')
if SMOKE_TEST and EVAL_SPLIT != 'dev':
    raise ValueError('La corrida smoke usa dev; test se reserva para una corrida full.')
environment.summary()

ENVIRONMENT
os             : Linux 6.6.159+
python         : 3.13.16
interpreter    : /usr/bin/python3
colab          : True
logical_cores  : 2
torch          : 2.11.0+cu130
cuda_available : True
gpu            : Tesla T4
gpu_memory_gb  : 15.64
ram_total_gb   : 13.6
ram_free_gb    : 6.3
HF_HOME        : /content/drive/MyDrive/projecto_rag/hf_cache
jax_guard      : True
ca_bundle      : not present (normal)
artifacts      : -> /content/drive/MyDrive/projecto_rag/artifacts (Drive)


## 2. Funciones de recuperación y medición

Esta celda es una copia del módulo local, no depende de haberlo subido al repositorio remoto.
El hash del código forma parte de la procedencia de la corrida.

In [43]:
"""Etapa 3 inicial: BM25, chunks estructurales y entrega por tokens.

"""
from __future__ import annotations

from dataclasses import asdict, dataclass
from datetime import datetime, timezone
from pathlib import Path
import hashlib
import importlib.metadata
import json
import math
import random
import re
import time
import unicodedata


STOP = set(
    "de la que el en y los se del las un por con no una su para "
    "es al lo como más a o sus le les".split()
)
EVAL_FILES = ("single_turn.jsonl", "dialogues.jsonl", "scenarios.jsonl")


def canonical_hash(value):
    blob = json.dumps(value, sort_keys=True, ensure_ascii=False, separators=(",", ":"))
    return hashlib.sha256(blob.encode("utf-8")).hexdigest()


def file_hash(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()


def read_jsonl(path):
    return [json.loads(line) for line in Path(path).read_text(encoding="utf-8").splitlines() if line.strip()]


def write_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(value, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")


def write_jsonl(path, rows):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text("".join(json.dumps(r, ensure_ascii=False) + "\n" for r in rows), encoding="utf-8")


def verify_frozen_files(eval_dir):
    """Verifica los tres conjuntos, sin escribir ni volver a congelar."""
    eval_dir = Path(eval_dir)
    frozen = json.loads((eval_dir / "FROZEN.json").read_text(encoding="utf-8"))
    for name in EVAL_FILES:
        rows = read_jsonl(eval_dir / name)
        spec = frozen["files"][name]
        if len(rows) != spec["rows"] or canonical_hash(rows) != spec["sha256_canonical"]:
            raise ValueError(f"Evaluación modificada: {name}. No se recalculó FROZEN.json.")
    return frozen


@dataclass(frozen=True)
class RetrievalConfig:
    name: str = "bm25"
    child_tokens: int = 320
    overlap_fraction: float = 0.15
    budget_tokens: int = 2500
    candidate_k: int = 100
    metric_ks: tuple = (1, 3, 5)
    bm25_k1: float = 1.5
    bm25_b: float = 0.75
    seed: int = 1337
    bootstrap_samples: int = 2000

    def __post_init__(self):
        if self.name not in ("bm25", "dense_bge_m3", "hybrid_rrf"):
            raise ValueError("Brazo no reconocido.")
        if self.child_tokens < 32 or not 0 <= self.overlap_fraction < 0.5:
            raise ValueError("child_tokens >= 32 y overlap_fraction en [0, 0.5).")
        if self.budget_tokens < 1 or self.candidate_k < 1:
            raise ValueError("Presupuesto y candidatos deben ser positivos.")
        if not self.metric_ks or min(self.metric_ks) < 1 or max(self.metric_ks) > self.candidate_k:
            raise ValueError("metric_ks debe estar entre 1 y candidate_k.")
        if self.bootstrap_samples < 1 or self.bm25_k1 <= 0 or not 0 <= self.bm25_b <= 1:
            raise ValueError("Parámetros BM25 o bootstrap inválidos.")


def token_count(tokenizer, text):
    return len(tokenizer.encode(text, add_special_tokens=False))


def chunk_documents(documents, tokenizer, cfg):
    """Offsets del texto original; nunca mezcla secciones ni decodifica tokens cortados.

    Dimensiona en el tokenizer del reranker, prefijo incluido. La entrega se cuenta
    aparte en el tokenizer de Qwen. No carga los pesos de ningún modelo.
    """
    chunks = []
    doc_ids = set()
    for doc in sorted(documents, key=lambda d: d["doc_id"]):
        doc_id = doc["doc_id"]
        if doc_id in doc_ids:
            raise ValueError(f"doc_id duplicado: {doc_id}")
        doc_ids.add(doc_id)
        metadata = doc.get("metadata", {})
        seen_sections = set()
        for sec in sorted(doc["sections"], key=lambda s: s["start"]):
            sec_id = sec["section_id"]
            if sec_id in seen_sections:
                raise ValueError(f"section_id duplicado: {sec_id}")
            seen_sections.add(sec_id)
            if not 0 <= sec["start"] < sec["end"] <= len(doc["text"]):
                raise ValueError(f"Offsets inválidos: {sec_id}")
            body = doc["text"][sec["start"]:sec["end"]]
            if not body.strip():
                continue
            # Prefijo mínimo fijo: identificador normativo y título de la sección.
            # No recorta títulos o texto jurídico en silencio.
            prefix = f"{doc_id} | {sec['title']}\n"
            prefix_n = token_count(tokenizer, prefix)
            room = cfg.child_tokens - prefix_n
            if room < 16:
                raise ValueError(f"Prefijo demasiado largo para el chunk: {sec_id}")
            encoded = tokenizer(body, add_special_tokens=False, return_offsets_mapping=True, truncation=False)
            offsets = encoded["offset_mapping"]
            if not offsets:
                raise ValueError(f"Texto no vacío sin tokens: {sec_id}")
            start_i = 0
            while start_i < len(offsets):
                end_i = min(start_i + room, len(offsets))
                start_char = 0 if start_i == 0 else offsets[start_i][0]
                end_char = len(body) if end_i == len(offsets) else offsets[end_i][0]
                text = body[start_char:end_char]
                # Retokenizar el fragmento aislado puede cambiar sus bordes BPE.
                while token_count(tokenizer, prefix + text) > cfg.child_tokens:
                    end_i -= 1
                    if end_i <= start_i:
                        raise ValueError(f"No se pudo dimensionar un fragmento: {sec_id}")
                    end_char = offsets[end_i][0]
                    text = body[start_char:end_char]
                if end_char <= start_char:
                    raise ValueError(f"Fragmento sin avance: {sec_id}")
                if text.strip():
                    absolute_start, absolute_end = sec["start"] + start_char, sec["start"] + end_char
                    identity = {"section_id": sec_id, "start": absolute_start, "end": absolute_end, "text": text}
                    chunks.append({
                        "chunk_id": f"{sec_id}@{absolute_start}:{absolute_end}-{canonical_hash(identity)[:12]}",
                        "doc_id": doc_id, "section_id": sec_id,
                        "title": doc["title"], "section_title": sec["title"],
                        "start": absolute_start, "end": absolute_end,
                        "text": text, "prefix": prefix, "index_text": prefix + text,
                        "reranker_tokens": token_count(tokenizer, prefix + text),
                        "metadata": metadata,
                    })
                if end_i == len(offsets):
                    break
                overlap = int((end_i - start_i) * cfg.overlap_fraction)
                start_i = max(start_i + 1, end_i - overlap)
    if not chunks:
        raise ValueError("El corpus no produjo chunks.")
    if len({c["chunk_id"] for c in chunks}) != len(chunks):
        raise ValueError("IDs de chunk duplicados.")
    return chunks


def validate_gold_in_index(items, chunks):
    available = {(c["doc_id"], c["section_id"]) for c in chunks}
    missing = []
    for item in items:
        if item["answerable"] and not item["gold"]:
            raise ValueError(f"Ítem respondible sin gold: {item['item_id']}")
        for gold in item["gold"]:
            pair = (gold["doc_id"], gold["section_id"])
            if pair not in available:
                missing.append((item["item_id"], pair))
    if missing:
        raise ValueError(f"Gold ausente del índice: {missing[:10]}")


class BM25Retriever:
    def __init__(self, chunks, cfg):
        import bm25s
        import Stemmer
        self.bm25s = bm25s
        self.stemmer = Stemmer.Stemmer("spanish")
        self.cfg = cfg
        self.chunks = chunks
        texts = [" ".join(self.words(c["index_text"])) for c in chunks]
        tokens = self.tokenize(texts)
        self.index = bm25s.BM25(k1=cfg.bm25_k1, b=cfg.bm25_b, method="lucene")
        self.index.index(tokens, show_progress=False)

    def words(self, text):
        text = unicodedata.normalize("NFC", text.lower())
        text = re.sub(r"\b(\d{2})\.(\d{3})\b", r"\1\2", text)
        return [self.stemmer.stemWord(w) for w in re.findall(r"\w+", text) if w not in STOP]

    def tokenize(self, texts):
        return self.bm25s.tokenize(texts, stopwords=[], token_pattern=r"(?u)\b\w+\b", show_progress=False)

    def retrieve(self, query):
        words = self.words(query)
        if not words:
            return []
        indices, scores = self.index.retrieve(
            self.tokenize([" ".join(words)]),
            k=min(self.cfg.candidate_k, len(self.chunks)), show_progress=False,
        )
        ranked = [
            {**self.chunks[int(i)], "score": float(score)}
            for i, score in zip(indices[0], scores[0]) if float(score) > 0
        ]
        return sorted(ranked, key=lambda c: (-c["score"], c["chunk_id"]))


def passage_block(chunk):
    # Este mismo bloque se entrega a la etapa 4, incluidos sus identificadores.
    return f"[{chunk['doc_id']} · {chunk['section_id']} · {chunk['chunk_id']}]\n{chunk['index_text']}"


def render_context(chunks):
    return "\n\n".join(passage_block(c) for c in chunks)


def deliver_by_budget(ranked, tokenizer, budget):
    delivered, skipped = [], []
    for c in ranked:
        if token_count(tokenizer, render_context(delivered + [c])) <= budget:
            delivered.append(c)
        else:
            skipped.append(c["chunk_id"])
    used = token_count(tokenizer, render_context(delivered))
    if used > budget:
        raise AssertionError("La entrega excedió el presupuesto.")
    return delivered, skipped, used


def normalize_answer(text):
    text = unicodedata.normalize("NFD", text.lower())
    text = "".join(c for c in text if unicodedata.category(c) != "Mn")
    text = re.sub(r"\(\s*\d+\s*\)", " ", text)
    text = re.sub(r"(?<=\d)[.,\s](?=\d{3}(?:\D|$))", "", text)
    text = re.sub(r"[^\w\s]", " ", text)
    return " ".join(text.split())


def contains_key(text, keys):
    normalized = f" {normalize_answer(text)} "
    return any(
        bool(normalize_answer(key)) and f" {normalize_answer(key)} " in normalized
        for key in keys or []
    )


def score_retrieval(item, ranked, delivered, ks):
    """k = posiciones de chunks recuperados; MRR truncado a candidate_k.

    Fuera del corpus: métricas de recuperación nulas, nunca fallos artificiales.
    answer_delivered busca claves SOLO en cuerpos, no en títulos/IDs del prefijo.
    """
    metrics = {"max_score": ranked[0]["score"] if ranked else 0.0}
    gold = {(g["doc_id"], g["section_id"]) for g in item["gold"]}
    gold_docs = {g[0] for g in gold}
    answerable = item["answerable"]
    for k in ks:
        top = ranked[:k]
        metrics[f"doc_hit@{k}"] = int(any(c["doc_id"] in gold_docs for c in top)) if answerable else None
        metrics[f"section_hit@{k}"] = int(any((c["doc_id"], c["section_id"]) in gold for c in top)) if answerable else None
    for level in ["doc", "section"]:
        position = next((i for i, c in enumerate(ranked, 1) if (
            c["doc_id"] in gold_docs if level == "doc" else (c["doc_id"], c["section_id"]) in gold
        )), None)
        metrics[f"{level}_mrr_candidates"] = (1.0 / position if position else 0.0) if answerable else None
    metrics["delivered_section_hit"] = int(any((c["doc_id"], c["section_id"]) in gold for c in delivered)) if answerable else None
    keys = item.get("answer_key")
    metrics["answer_delivered"] = int(any(contains_key(c["text"], keys) for c in delivered)) if answerable and keys else None
    return metrics


def reranker_overflow(query, candidates, tokenizer, max_length=512):
    lengths = [len(tokenizer(query, c["index_text"], add_special_tokens=True, truncation=False)["input_ids"]) for c in candidates]
    return {"pairs": len(lengths), "over_limit": sum(n > max_length for n in lengths), "max_pair_tokens": max(lengths, default=0)}


def evaluate_items(items, retriever, generator_tokenizer, reranker_tokenizer, run_spec):
    run_key = canonical_hash(run_spec)
    rows = []
    cfg = retriever.cfg
    for item in items:
        t0 = time.perf_counter()
        ranked = retriever.retrieve(item["question"])
        search_s = time.perf_counter() - t0
        t1 = time.perf_counter()
        delivered, skipped, used = deliver_by_budget(ranked, generator_tokenizer, cfg.budget_tokens)
        packing_s = time.perf_counter() - t1
        rows.append({
            "run_key": run_key, "run_spec": run_spec,
            "item_id": item["item_id"], "question": item["question"],
            "category": item["category"], "split": item["split"],
            "answerable": item["answerable"], "gold": item["gold"],
            "answer_key": item.get("answer_key"), "expected_verdict": item.get("expected_verdict"),
            "config": asdict(cfg),
            "metrics": score_retrieval(item, ranked, delivered, cfg.metric_ks),
            "tokens_delivered": used, "budget_tokens": cfg.budget_tokens,
            "search_s": search_s, "packing_s": packing_s, "retrieval_s": search_s + packing_s,
            "reranker_window_check": reranker_overflow(item["question"], ranked, reranker_tokenizer),
            "candidates": [{k: c[k] for k in ["chunk_id", "doc_id", "section_id", "score"]} for c in ranked],
            "delivered": [{**c, "passage": passage_block(c)} for c in delivered],
            "skipped_chunk_ids": skipped,
        })
    return rows


def bootstrap_mean(values, seed=1337, samples=2000):
    if not values:
        return None, None
    rng = random.Random(seed)
    means = sorted(sum(rng.choices(values, k=len(values))) / len(values) for _ in range(samples))
    return means[int(0.025 * (samples - 1))], means[int(0.975 * (samples - 1))]


def summary_rows(rows, cfg):
    result = []
    for category in ["all"] + sorted({r["category"] for r in rows}):
        group = [r for r in rows if category == "all" or r["category"] == category]
        if not group:
            continue
        for metric in rows[0]["metrics"]:
            if metric == "max_score":
                continue  # Los puntajes OOC se inspeccionan aparte; no son exactitud.
            values = [r["metrics"][metric] for r in group if r["metrics"][metric] is not None]
            low, high = bootstrap_mean(values, cfg.seed, cfg.bootstrap_samples)
            result.append({"category": category, "metric": metric, "n": len(values),
                           "mean": sum(values) / len(values) if values else None,
                           "ci95_low": low, "ci95_high": high})
    return result


def paired_comparison(previous, current, metric="answer_delivered"):
    """Para futuras filas: mismos ítems, split, benchmark y presupuesto."""
    a, b = {r["item_id"]: r for r in previous}, {r["item_id"]: r for r in current}
    if len(a) != len(previous) or len(b) != len(current) or a.keys() != b.keys():
        raise ValueError("La comparación requiere los mismos ítems, sin duplicados.")
    wins = losses = ties = 0
    differences = []
    for ident in sorted(a):
        x, y = a[ident], b[ident]
        if (x["split"] != y["split"] or x["budget_tokens"] != y["budget_tokens"]
                or x["run_spec"]["eval"] != y["run_spec"]["eval"]
                or x["run_spec"]["corpus_hash"] != y["run_spec"]["corpus_hash"]):
            raise ValueError("Evaluación, corpus, split o presupuesto diferentes.")
        p, q = x["metrics"][metric], y["metrics"][metric]
        if p is None or q is None:
            continue
        if p not in (0, 1) or q not in (0, 1):
            raise ValueError("El test de signos requiere una métrica binaria.")
        wins += q > p
        losses += q < p
        ties += q == p
        differences.append(q - p)
    n = wins + losses
    p_value = min(1.0, 2 * sum(math.comb(n, i) for i in range(min(wins, losses) + 1)) / 2**n) if n else 1.0
    low, high = bootstrap_mean(differences)
    return {"metric": metric, "wins": wins, "losses": losses, "ties": ties,
            "sign_test_p_two_sided": p_value, "delta_ci95": [low, high]}


def package_versions():
    versions = {}
    for name in ["bm25s", "PyStemmer", "transformers", "tokenizers", "numpy", "scipy", "huggingface-hub"]:
        try:
            versions[name] = importlib.metadata.version(name)
        except importlib.metadata.PackageNotFoundError:
            versions[name] = None
    return versions


def project_sources():
    """Huellas de los auxiliares cargados, además del código de esta etapa."""
    import sys
    sources = {}
    for name in ["environment", "schema", "console", "store", "llm"]:
        module = sys.modules.get(name)
        path = getattr(module, "__file__", None)
        if path and Path(path).is_file():
            sources[name] = {"sha256": file_hash(path)}
    return sources


def save_index(retriever, index_dir, manifest):
    index_dir = Path(index_dir)
    index_dir.mkdir(parents=True, exist_ok=True)
    existing = index_dir / "manifest.json"
    if existing.exists() and canonical_hash(json.loads(existing.read_text(encoding="utf-8"))) != canonical_hash(manifest):
        raise ValueError("El directorio de índice pertenece a otra configuración.")
    retriever.index.save(str(index_dir / "bm25"))
    write_jsonl(index_dir / "chunks.jsonl", retriever.chunks)
    write_json(existing, manifest)


def utc_now():
    return datetime.now(timezone.utc).isoformat()

In [44]:
IMPLEMENTATION_SHA256 = 'db79824cd1794246673485c1a9d2835ed0b3bcf14a96301053c52a3312617060'
CFG = RetrievalConfig(seed=SEED)
print(asdict(CFG))

{'name': 'bm25', 'child_tokens': 320, 'overlap_fraction': 0.15, 'budget_tokens': 2500, 'candidate_k': 100, 'metric_ks': (1, 3, 5), 'bm25_k1': 1.5, 'bm25_b': 0.75, 'seed': 1337, 'bootstrap_samples': 2000}


## 3. Validar corpus y congelamiento

Se verifica la evaluación completa contra el corpus completo antes de seleccionar dev o smoke.
Si hay varias copias distintas del conjunto congelado, se detiene para que elijas `EVAL_DIR`.
Si faltan insumos, ejecutar etapa 1 y traer la carpeta congelada; no reconstruir etiquetas aquí.

In [45]:
import schema
from schema import Document

if not DOCUMENTS_PATH.is_file():
    raise FileNotFoundError(
        f'Falta {DOCUMENTS_PATH}. Ejecutá 01_documents.ipynb en el mismo Drive '
        'o recuperá documents.jsonl; no se genera un corpus alternativo.'
    )

if EVAL_DIR is None:
    options = [Path('eval'), Path('etapa2_congelada/eval'),
               DOCUMENTS_PATH.parent / 'etapa2_congelada/eval']
    existing = list(dict.fromkeys(p.resolve() for p in options if (p / 'FROZEN.json').is_file()))
    if not existing:
        raise FileNotFoundError(
            'Falta la evaluación congelada. Subí etapa2_congelada/eval al proyecto '
            'o fijá EVAL_DIR a la carpeta que contiene FROZEN.json y los tres JSONL.'
        )
    versions = {canonical_hash(verify_frozen_files(p)) for p in existing}
    if len(versions) > 1:
        raise RuntimeError(f'Hay evaluaciones congeladas diferentes: {existing}. Elegí EVAL_DIR.')
    EVAL_DIR = existing[0]
EVAL_DIR = Path(EVAL_DIR)
FROZEN = verify_frozen_files(EVAL_DIR)
schema.validate_documents(DOCUMENTS_PATH).show().raise_if_failed()
schema.load_eval(EVAL_DIR, documents_path=DOCUMENTS_PATH)
schema.validate_eval(EVAL_DIR, DOCUMENTS_PATH).show().raise_if_failed()

all_documents = read_jsonl(DOCUMENTS_PATH)
corpus_hash = schema.documents_hash([Document.model_validate(d) for d in all_documents])
if corpus_hash != FROZEN['documents_sha256']:
    raise ValueError('El hash del corpus difiere del usado para congelar la evaluación.')
all_items = read_jsonl(EVAL_DIR / 'single_turn.jsonl')
items = [q for q in all_items if q['split'] == EVAL_SPLIT]
if not items:
    raise ValueError(f'No hay preguntas para {EVAL_SPLIT}.')
documents = all_documents

if SMOKE_TEST:
    # Hasta dos preguntas por categoría; todos los gold y diez distractores con semilla.
    selected = []
    for category in sorted({q['category'] for q in items}):
        selected.extend([q for q in items if q['category'] == category][:2])
    items = selected
    gold_ids = {g['doc_id'] for q in items for g in q['gold']}
    others = sorted([d['doc_id'] for d in all_documents if d['doc_id'] not in gold_ids])
    sampled = set(random.Random(SEED).sample(others, min(10, len(others))))
    documents = [d for d in all_documents if d['doc_id'] in gold_ids | sampled]

pending = sum('pendiente' in q.get('notes', '').lower() for q in items)
if pending:
    warnings.warn(f'{pending}/{len(items)} preguntas aún declaran revisión humana pendiente. '
                  'Los resultados deben considerarse provisionales.')
print('Perfil:', PROFILE, '| split:', EVAL_SPLIT)
print('Evaluación:', EVAL_DIR)
print('Documentos indexados:', len(documents), '/', len(all_documents))
print('Preguntas seleccionadas:', len(items))
print('Hash del corpus completo:', corpus_hash)

documents (documents.jsonl): OK
   documents: 170
   sections: 7070
   characters: 10685267
   with version: 0
   with date: 170
eval set (/content/projecto_rag/eval): OK
   single-turn items: 30
   by split: {'dev': 7, 'test': 23}
   by category: {'topical': 15, 'metadata': 7, 'identifier': 5, 'out_of_corpus': 3}
   dialogues: 8
   dialogue turns: 32
   scenarios: 3
   probes by ability: {'extraction': 3, 'multi_session': 6, 'temporal': 3, 'knowledge_update': 9, 'preference': 3, 'abstention': 3}
   yes/no rows (expected_verdict): 12
Perfil: full | split: test
Evaluación: /content/projecto_rag/eval
Documentos indexados: 170 / 170
Preguntas seleccionadas: 23
Hash del corpus completo: 519330293d3b792c42325b784acc17b0495319994dcacaf90021b5f08a435a59


/tmp/ipykernel_803/535118348.py:52: UserWarning: 23/23 preguntas aún declaran revisión humana pendiente. Los resultados deben considerarse provisionales.
  warnings.warn(f'{pending}/{len(items)} preguntas aún declaran revisión humana pendiente. '


## 4. Tokenizers abiertos con revisión fijada

Se utiliza el tokenizer de `cross-encoder/mmarco-mMiniLMv2-L12-H384-v1` para dimensionar
hijos y comprobar pares de 512 tokens; no se ejecuta un reranker en esta línea de base.
El tokenizer de `Qwen/Qwen3-8B` cuenta la entrega. No se descargan pesos de los modelos.
La primera ejecución resuelve las revisiones y las guarda; las siguientes reutilizan esos commits.

In [46]:
from huggingface_hub import snapshot_download
from transformers import AutoTokenizer

tokenizer_models = {
    'generator': 'Qwen/Qwen3-8B',
    'reranker': 'cross-encoder/mmarco-mMiniLMv2-L12-H384-v1',
}
lock_path = ART / 'index' / 'stage3_tokenizer_revisions.json'
locks = json.loads(lock_path.read_text(encoding='utf-8')) if lock_path.exists() else {}
tokenizers = {}
for role, model in tokenizer_models.items():
    if role in locks and locks[role]['model'] != model:
        raise ValueError('El lock de tokenizers pertenece a otros modelos.')
    revision = locks.get(role, {}).get('revision', 'main')
    snapshot = Path(snapshot_download(
        repo_id=model, revision=revision,
        allow_patterns=['tokenizer.json', 'tokenizer_config.json', 'special_tokens_map.json',
                        'vocab.txt', 'vocab.json', 'merges.txt', 'config.json', 'added_tokens.json'],
    ))
    resolved = snapshot.name
    if not re.fullmatch(r'[0-9a-f]{40}', resolved):
        raise RuntimeError(f'No se pudo identificar el commit del tokenizer: {snapshot}')
    tokenizers[role] = AutoTokenizer.from_pretrained(snapshot, local_files_only=True, use_fast=True)
    locks[role] = {'model': model, 'revision': resolved}
write_json(lock_path, locks)
GEN_TOKENIZER = tokenizers['generator']
RERANK_TOKENIZER = tokenizers['reranker']
print(json.dumps(locks, indent=2))

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

{
  "generator": {
    "model": "Qwen/Qwen3-8B",
    "revision": "b968826d9c46dd6066d109eabc6255188de91218"
  },
  "reranker": {
    "model": "cross-encoder/mmarco-mMiniLMv2-L12-H384-v1",
    "revision": "1427fd652930e4ba29e8149678df786c240d8825"
  }
}


## 5. Construir índice y comprobar gold

El manifiesto identifica corpus, documentos efectivamente indexados, tokenizers, configuración,
normalización y código. Cada cambio obtiene su propio directorio: no se reutiliza un índice incompatible.
La construcción BM25 se repite al ejecutar esta celda; se guarda el índice para las etapas siguientes.

In [47]:
from console import StageRun

run = StageRun.start('03_retrieval', ART, profile=PROFILE, config=asdict(CFG))
started_at = utc_now()
build_start = time.perf_counter()
chunks = chunk_documents(documents, RERANK_TOKENIZER, CFG)
validate_gold_in_index(items, chunks)
retriever = BM25Retriever(chunks, CFG)
build_s = time.perf_counter() - build_start

index_manifest = {
    'stage': '03_retrieval', 'profile': PROFILE,
    'corpus_hash': corpus_hash,
    'indexed_documents_hash': canonical_hash(sorted(documents, key=lambda d: d['doc_id'])),
    'documents': len(documents), 'chunks': len(chunks),
    'chunks_hash': canonical_hash(chunks),
    'config': asdict(CFG), 'tokenizers': locks,
    'lexical': {'method': 'lucene', 'stemmer': 'spanish', 'stopwords': sorted(STOP),
                'identifier_normalization': '24.240 -> 24240',
                'prefix': 'doc_id | section_title'},
    'implementation_sha256': IMPLEMENTATION_SHA256, 'packages': package_versions(),
    'project_sources': project_sources(),
}
INDEX_KEY = canonical_hash(index_manifest)
INDEX_DIR = ART / 'index' / f'stage3_{INDEX_KEY[:16]}'
save_index(retriever, INDEX_DIR, index_manifest)
RUN_SPEC = {
    'stage': '03_retrieval', 'profile': PROFILE, 'split': EVAL_SPLIT,
    'config': asdict(CFG), 'eval': FROZEN, 'corpus_hash': corpus_hash,
    'index_key': INDEX_KEY, 'item_ids': [q['item_id'] for q in items],
    'implementation_sha256': IMPLEMENTATION_SHA256,
}
RUN_KEY = canonical_hash(RUN_SPEC)
RESULTS_DIR = ART / 'results' / f'stage3_{RUN_KEY[:16]}'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
print('Chunks:', len(chunks), '| máximo:', max(c['reranker_tokens'] for c in chunks))
print('Construcción (s):', round(build_s, 2))
print('Índice:', INDEX_DIR)

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (14990 > 512). Running this sequence through the model will result in indexing errors


== 03_retrieval · profile full · report -> /content/projecto_rag/artifacts/full/run_report.json


DEBUG:bm25s:Building index from IDs objects


Chunks: 13630 | máximo: 320
Construcción (s): 35.57
Índice: /content/projecto_rag/artifacts/full/index/stage3_36a95c9c1551289f


## 6. Guardas que deben fallar antes de medir

Una referencia gold inexistente debe detener la evaluación. Un pasaje que no entra debe saltarse,
sin impedir entregar otro más pequeño que venga después. Estas pruebas no modifican los insumos.

In [48]:
try:
    validate_gold_in_index([{
        'item_id': 'smoke_missing_gold', 'answerable': True,
        'gold': [{'doc_id': '__missing__', 'section_id': '__missing__#ART-1'}]
    }], chunks)
except ValueError:
    print('OK: un gold ausente se rechaza.')
else:
    raise AssertionError('No saltó la guarda de gold.')

small = {**chunks[0], 'chunk_id': 'guard_small', 'text': 'dato breve', 'index_text': 'dato breve'}
large = {**small, 'chunk_id': 'guard_large', 'text': 'dato largo ' * 100,
         'index_text': 'dato largo ' * 100}
guard_budget = token_count(GEN_TOKENIZER, render_context([small]))
kept, skipped, used = deliver_by_budget([large, small], GEN_TOKENIZER, guard_budget)
assert [c['chunk_id'] for c in kept] == ['guard_small'] and 'guard_large' in skipped
assert used <= guard_budget
print('OK: se salta un pasaje demasiado largo y se considera el siguiente.')

OK: un gold ausente se rechaza.
OK: se salta un pasaje demasiado largo y se considera el siguiente.


## 7. Medir y guardar resultados

`hit@k` usa las primeras k posiciones de chunks. MRR se trunca a los candidatos recuperados.
`delivered_section_hit` evalúa lo que realmente entró. `answer_delivered` es una aproximación léxica:
busca alguna clave en el cuerpo de un pasaje entregado, sin contar los identificadores del prefijo.
No certifica una respuesta correcta ni la vigencia jurídica. Fuera del corpus, estas métricas son nulas.

El contexto exacto queda en `contexts.jsonl`, listo para el ensamblador de etapa 4. El límite se aplica al
bloque completo de pasajes, sin cortar un fragmento para hacerlo entrar. Se consideran hasta 100 candidatos;
se registra el presupuesto realmente usado, aunque la lista no lo llene.

In [49]:
rows = evaluate_items(items, retriever, GEN_TOKENIZER, RERANK_TOKENIZER, RUN_SPEC)
summary = summary_rows(rows, CFG)
write_jsonl(RESULTS_DIR / 'retrieval.jsonl', rows)
write_jsonl(RESULTS_DIR / 'summary.jsonl', summary)
write_json(RESULTS_DIR / 'run_spec.json', RUN_SPEC)
write_jsonl(RESULTS_DIR / 'contexts.jsonl', [
    {'run_key': RUN_KEY, 'item_id': r['item_id'], 'question': r['question'],
     'tokens_delivered': r['tokens_delivered'],
     'context': render_context(r['delivered']), 'passages': r['delivered']}
    for r in rows
])

import csv
with (RESULTS_DIR / 'summary.csv').open('w', encoding='utf-8', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=list(summary[0]))
    writer.writeheader()
    writer.writerows(summary)

try:
    import pandas as pd
except ImportError:
    for r in summary:
        print(r)
else:
    display(pd.DataFrame(summary))
print('Promedio de tokens entregados:', round(sum(r['tokens_delivered'] for r in rows) / len(rows), 1))
print('Resultados:', RESULTS_DIR)

,category,metric,n,mean,ci95_low,ci95_high
0,all,doc_hit@1,21,0.857143,0.714286,1.000000
1,all,section_hit@1,21,0.761905,0.571429,0.952381
2,all,doc_hit@3,21,0.952381,0.857143,1.000000
3,all,section_hit@3,21,0.857143,0.666667,1.000000
4,all,doc_hit@5,21,0.952381,0.857143,1.000000
5,all,section_hit@5,21,0.857143,0.666667,1.000000
6,all,doc_mrr_candidates,21,0.900794,0.781746,1.000000
7,all,section_mrr_candidates,21,0.820635,0.663492,0.955556
8,all,delivered_section_hit,21,0.904762,0.761905,1.000000
9,all,answer_delivered,21,0.952381,0.857143,1.000000


Promedio de tokens entregados: 2456.8
Resultados: /content/projecto_rag/artifacts/full/results/stage3_67ecda17a40b8c18


## 8. Abrir la caja de los errores

Inspeccionar preguntas que no recibieron la sección o cuya clave no llegó, aunque se recuperó el documento.
No corregir gold automáticamente a partir de este ranking. Para test, documentar los fallos sin ajustar el pipeline.
El chequeo de ventana del reranker es preventivo: esta fila no lo ejecuta.

In [50]:
for r in rows:
    if r['answerable'] and (not r['metrics']['delivered_section_hit'] or not r['metrics']['answer_delivered']):
        print('\n', r['item_id'], r['question'])
        print('Gold:', r['gold'])
        print('Métricas:', r['metrics'])
        print('Top 5:', [(c['section_id'], round(c['score'], 3)) for c in r['candidates'][:5]])
        print('Entregados:', [c['section_id'] for c in r['delivered']])

pair_count = sum(r['reranker_window_check']['pairs'] for r in rows)
overflow = sum(r['reranker_window_check']['over_limit'] for r in rows)
print('Pares consulta+hijo >512:', overflow, '/', pair_count)
print('Puntajes máximos; no equivalen a probabilidades de respuesta:')
for r in rows:
    print(r['item_id'], r['category'], r['metrics']['max_score'])


 q004 ¿Las características prometidas en una publicidad obligan al oferente aunque no se repitan en el contrato?
Gold: [{'doc_id': 'LEY-24240', 'section_id': 'LEY-24240#ART-8'}]
Métricas: {'max_score': 7.440732955932617, 'doc_hit@1': 0, 'section_hit@1': 0, 'doc_hit@3': 0, 'section_hit@3': 0, 'doc_hit@5': 0, 'section_hit@5': 0, 'doc_mrr_candidates': 0.08333333333333333, 'section_mrr_candidates': 0.06666666666666667, 'delivered_section_hit': 0, 'answer_delivered': 1}
Top 5: [('LEY-26994#ANEXO-I/ART-1526', 7.441), ('LEY-26994#ANEXO-I/ART-1103', 6.521), ('LEY-26994#ANEXO-I/ART-1252', 6.02), ('DEC-92-1997#ANEXO-XII/ART-6-H9377efad5af6', 5.919), ('LEY-26994#ANEXO-I/ART-1803', 5.786)]
Entregados: ['LEY-26994#ANEXO-I/ART-1526', 'LEY-26994#ANEXO-I/ART-1103', 'LEY-26994#ANEXO-I/ART-1252', 'DEC-92-1997#ANEXO-XII/ART-6-H9377efad5af6', 'LEY-26994#ANEXO-I/ART-1803', 'LEY-26994#ANEXO-I/ART-1008', 'LEY-26994#ANEXO-I/ART-1026', 'LEY-26994#ANEXO-I/ART-1352', 'LEY-26994#ANEXO-I/ART-1108', 'LEY-26994#ANE

## 9. Guardar la línea de base BM25

Esta corrida conserva el índice de la implementación original. Los resultados sobre `dev` se usarán para comparaciones pareadas; `test` queda reservado. La métrica `answer_delivered` busca una coincidencia lexical, no juzga corrección semántica.


In [51]:
report = {
    'stage': '03_retrieval', 'status': 'ok', 'profile': PROFILE, 'split': EVAL_SPLIT,
    'started_at': started_at, 'finished_at': utc_now(), 'run_key': RUN_KEY,
    'run_spec': RUN_SPEC, 'index_dir': str(INDEX_DIR), 'results_dir': str(RESULTS_DIR),
    'documents': len(documents), 'chunks': len(chunks), 'items': len(rows),
    'index_build_s': build_s, 'llm_calls': 0,
    'mean_tokens_delivered': sum(r['tokens_delivered'] for r in rows) / len(rows),
    'reranker_pairs_over_limit': overflow, 'reranker_pairs_checked': pair_count,
    'human_review_pending_in_notes': pending,
    'limitations': ['BM25 only', 'smoke is not a benchmark' if SMOKE_TEST else 'single row; no improvement claim',
                    'answer_delivered is lexical containment; not semantic correctness'],
}
write_json(RESULTS_DIR / 'run_report.json', report)
run.note('documents', len(documents))
run.note('chunks', len(chunks))
run.note('items', len(rows))
run.finish()
print('Corrida finalizada:', RUN_KEY)
print('Para dev completo: SMOKE_TEST=False, EVAL_SPLIT="dev".')
print('Para reporte final, una vez fijadas las decisiones: SMOKE_TEST=False, EVAL_SPLIT="test".')

== 03_retrieval finished: ok in 54.4 s
Corrida finalizada: 67ecda17a40b8c1805b806def23f5eac295e9eab5a52356b582194d05861c297
Para dev completo: SMOKE_TEST=False, EVAL_SPLIT="dev".
Para reporte final, una vez fijadas las decisiones: SMOKE_TEST=False, EVAL_SPLIT="test".


## 10. Recuperación densa con BGE-M3

**Hipótesis:** los embeddings densos pueden recuperar secciones reformuladas con términos distintos de los del documento. Mantendremos **exactamente los mismos chunks** y el mismo presupuesto que en BM25 para que la comparación no confunda el método con el tamaño de la evidencia.

El embedding de los documentos incorpora un **prefijo contextual** (identificador, título de documento, sección y metadatos disponibles). Como esto no coincide con el prefijo mínimo lexical original, su efecto se documenta como una decisión del brazo denso; no se le atribuye causalmente solo al modelo BGE-M3. Si se desea aislar el efecto del prefijo, haría falta un brazo denso sin prefijo adicional.

La matriz se cachea con una clave que incluye el hash de los textos que se embeben, la revisión del modelo y los parámetros. Así no se reutilizan embeddings de chunks distintos.

In [52]:
import numpy as np
import torch
from sentence_transformers import SentenceTransformer
from huggingface_hub import snapshot_download

DENSE_MODEL_ID = "BAAI/bge-m3"
DENSE_MAX_SEQ = 8192
DENSE_BATCH_SIZE = 8 if torch.cuda.is_available() else 2
DENSE_DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Dispositivo de embeddings:", DENSE_DEVICE)
if DENSE_DEVICE == "cpu":
    warnings.warn("BGE-M3 en CPU puede ser lento; se recomienda GPU T4 de Colab.")

# Metadatos no son evidencia nueva: sirven para desambiguar documentos cercanos.
def contextual_text(chunk):
    meta = chunk.get("metadata") or {}
    fields = [f"Documento: {chunk['title']}", f"ID: {chunk['doc_id']}",
              f"Sección: {chunk['section_title']}"]
    for key in ("version", "year", "date", "status"):
        value = meta.get(key)
        if value is not None and value != "":
            fields.append(f"{key}: {value}")
    fields.append(chunk["text"])
    return "\n".join(fields)

DENSE_TEXTS = [contextual_text(c) for c in chunks]
DENSE_REVISION = (snapshot_download(repo_id=DENSE_MODEL_ID, allow_patterns=[
    "config.json", "modules.json", "sentence_bert_config.json", "1_Pooling/config.json",
    "config_sentence_transformers.json", "README.md"
])).split("/")[-1]
if not re.fullmatch(r"[0-9a-f]{40}", DENSE_REVISION):
    raise RuntimeError("No se pudo fijar la revisión de BGE-M3.")
DENSE_SPEC = {"encoder": DENSE_MODEL_ID, "revision": DENSE_REVISION,
              "normalize_embeddings": True, "max_seq_length": DENSE_MAX_SEQ,
              "texts_sha256": canonical_hash(DENSE_TEXTS),
              "chunk_ids_sha256": canonical_hash([c['chunk_id'] for c in chunks]),
              "corpus_hash": corpus_hash, "profile": PROFILE}
DENSE_KEY = canonical_hash(DENSE_SPEC)
DENSE_DIR = ART / "embeddings" / f"bge_m3_{DENSE_KEY[:20]}"
DENSE_DIR.mkdir(parents=True, exist_ok=True)
MATRIX_FILE = DENSE_DIR / "matrix.npy"
if MATRIX_FILE.exists():
    old = json.loads((DENSE_DIR / "manifest.json").read_text(encoding="utf-8"))
    if old != DENSE_SPEC:
        raise RuntimeError("La caché de embeddings pertenece a otra especificación.")
    DENSE_MATRIX = np.load(MATRIX_FILE, allow_pickle=False)
    print("Embeddings recuperados de caché:", MATRIX_FILE)
else:
    dense_encoder = SentenceTransformer(DENSE_MODEL_ID, revision=DENSE_REVISION, device=DENSE_DEVICE,
                                       trust_remote_code=False)
    dense_encoder.max_seq_length = DENSE_MAX_SEQ
    DENSE_MATRIX = dense_encoder.encode(DENSE_TEXTS, batch_size=DENSE_BATCH_SIZE,
                                        show_progress_bar=True, normalize_embeddings=True,
                                        convert_to_numpy=True).astype(np.float32)
    np.save(MATRIX_FILE, DENSE_MATRIX, allow_pickle=False)
    write_json(DENSE_DIR / "manifest.json", DENSE_SPEC)
    print("Embeddings creados:", MATRIX_FILE)
if len(DENSE_MATRIX) != len(chunks):
    raise RuntimeError("La matriz densa no coincide con el número de chunks.")
if not np.isfinite(DENSE_MATRIX).all():
    raise RuntimeError("Se encontraron embeddings NaN/Inf.")
if "dense_encoder" not in globals():
    dense_encoder = SentenceTransformer(DENSE_MODEL_ID, revision=DENSE_REVISION, device=DENSE_DEVICE,
                                       trust_remote_code=False)
    dense_encoder.max_seq_length = DENSE_MAX_SEQ
print("Matriz:", DENSE_MATRIX.shape, "| especificación:", DENSE_KEY)

Dispositivo de embeddings: cuda


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

Embeddings recuperados de caché: /content/projecto_rag/artifacts/full/embeddings/bge_m3_1818bc1802f3cda68058/matrix.npy
Matriz: (13630, 1024) | especificación: 1818bc1802f3cda68058e376b1cb7722a387ed1091a24092a3659a8c4c11faf5


## 11. Tres brazos comparables y fusión RRF

**Brazo A — BM25:** búsqueda léxica ya implementada. **Brazo B — denso:** coseno entre vectores normalizados de BGE-M3. **Brazo C — híbrido:** fusión de rankings BM25 y denso usando *Reciprocal Rank Fusion* con `k=60`. El RRF usa posiciones, no combina puntajes heterogéneos.

Los brazos mantienen el mismo `candidate_k`, los mismos chunks y `budget_tokens=2500`. Si se solicita un componente no cargado, la búsqueda falla explícitamente. Cada resultado conserva el `chunk_id`, `doc_id` y `section_id`.

In [53]:
from dataclasses import replace

class DenseRetriever:
    def __init__(self, chunks, cfg, encoder, matrix):
        if encoder is None or matrix is None:
            raise RuntimeError("El brazo denso requiere encoder y matriz.")
        self.chunks, self.cfg, self.encoder, self.matrix = chunks, cfg, encoder, matrix

    def retrieve(self, query):
        query_vec = self.encoder.encode([query], normalize_embeddings=True,
                                        convert_to_numpy=True)[0].astype(np.float32)
        similarities = self.matrix @ query_vec
        k = min(self.cfg.candidate_k, len(self.chunks))
        candidates = np.argpartition(-similarities, k - 1)[:k]
        order = sorted(candidates, key=lambda i: (-float(similarities[i]), self.chunks[i]["chunk_id"]))
        return [{**self.chunks[int(i)], "score": float(similarities[i])} for i in order]


def rrf_fusion(rankings, k=60, limit=100):
    """RRF sobre ids únicos; empates resueltos determinísticamente."""
    if len(rankings) < 2 or any(r is None for r in rankings):
        raise RuntimeError("RRF requiere dos rankings cargados.")
    scores, by_id = {}, {}
    for ranking in rankings:
        for rank, passage in enumerate(ranking, start=1):
            ident = passage["chunk_id"]
            scores[ident] = scores.get(ident, 0.0) + 1.0 / (k + rank)
            by_id[ident] = passage
    selected = sorted(scores, key=lambda ident: (-scores[ident], ident))[:limit]
    return [{**by_id[i], "score": scores[i]} for i in selected]


class HybridRetriever:
    def __init__(self, lexical, dense, cfg, rrf_k=60):
        if lexical is None or dense is None:
            raise RuntimeError("El híbrido requiere canales BM25 y denso.")
        self.lexical, self.dense, self.cfg, self.rrf_k = lexical, dense, cfg, rrf_k

    def retrieve(self, query):
        return rrf_fusion([self.lexical.retrieve(query), self.dense.retrieve(query)],
                          k=self.rrf_k, limit=self.cfg.candidate_k)

# Una configuración por brazo, sin tocar los parámetros compartidos.
CFG_BM25 = CFG
CFG_DENSE = replace(CFG, name="dense_bge_m3")
CFG_HYBRID = replace(CFG, name="hybrid_rrf")
dense_retriever = DenseRetriever(chunks, CFG_DENSE, dense_encoder, DENSE_MATRIX)
hybrid_retriever = HybridRetriever(retriever, dense_retriever, CFG_HYBRID)
ARMS = {"bm25": retriever, "dense_bge_m3": dense_retriever, "hybrid_rrf": hybrid_retriever}

# Guardas mínimas: mismos ids, presupuesto y número de candidatos.
assert all(r.cfg.budget_tokens == CFG.budget_tokens for r in ARMS.values())
assert len({c['chunk_id'] for c in chunks}) == len(chunks)
probe = items[0]['question']
for name, arm in ARMS.items():
    top = arm.retrieve(probe)
    assert len({c['chunk_id'] for c in top}) == len(top)
    print(name, "=>", [(c['section_id'], round(c['score'], 3)) for c in top[:3]])

bm25 => [('LEY-26994#ANEXO-I/ART-1526', 7.441), ('LEY-26994#ANEXO-I/ART-1103', 6.521), ('LEY-26994#ANEXO-I/ART-1252', 6.02)]
dense_bge_m3 => [('LEY-26994#ANEXO-I/ART-1803', 0.642), ('LEY-24240#ART-8', 0.635), ('LEY-26994#ANEXO-I/ART-1103', 0.618)]
hybrid_rrf => [('LEY-26994#ANEXO-I/ART-1103', 0.032), ('LEY-26994#ANEXO-I/ART-1803', 0.032), ('LEY-24240#ART-8', 0.029)]


## 12. Evaluación de los tres brazos por pregunta y categoría

Se reutilizan `evaluate_items`, `score_retrieval` y `deliver_by_budget` del notebook original. Se reportan `hit@k` (documento y sección), MRR, sección gold entregada, clave presente en el cuerpo, tokens y latencia. En preguntas `out_of_corpus`, no se imputa un fallo de hit artificial: estas consultas se revisan por separado.

Los resultados se guardan en directorios específicos, junto con la configuración exacta del brazo y el hash del conjunto congelado.

In [54]:
import csv

ARM_ROWS = {}
ARM_SUMMARIES = {}
ARM_DIRS = {}

for arm_name, arm in ARMS.items():
    arm_spec = {**RUN_SPEC, "arm": arm_name,
                "dense_spec": DENSE_SPEC if arm_name != "bm25" else None,
                "rrf_k": 60 if arm_name == "hybrid_rrf" else None}
    key = canonical_hash(arm_spec)
    directory = ART / "results" / f"stage3_comparison_{arm_name}_{key[:16]}"
    directory.mkdir(parents=True, exist_ok=True)
    measured = rows if arm_name == "bm25" else evaluate_items(
        items, arm, GEN_TOKENIZER, RERANK_TOKENIZER, arm_spec)
    # Conservar explícitamente el brazo que produjo cada resultado.
    for record in measured:
        record['arm'] = arm_name
    per_category = summary_rows(measured, CFG)
    write_jsonl(directory / "retrieval.jsonl", measured)
    write_jsonl(directory / "summary.jsonl", per_category)
    write_json(directory / "run_spec.json", arm_spec)
    write_jsonl(directory / "contexts.jsonl", [
        {"item_id": r["item_id"], "question": r["question"], "arm": arm_name,
         "tokens_delivered": r["tokens_delivered"], "context": render_context(r["delivered"]),
         "passages": r["delivered"]} for r in measured])
    with (directory / "summary.csv").open("w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=list(per_category[0])); writer.writeheader(); writer.writerows(per_category)
    ARM_ROWS[arm_name], ARM_SUMMARIES[arm_name], ARM_DIRS[arm_name] = measured, per_category, directory
    print(arm_name, "|", len(measured), "preguntas |", directory)

try:
    import pandas as pd
    display(pd.DataFrame([{
        "brazo": arm,
        "doc_hit@5": next((s["mean"] for s in summary if s["category"] == "all" and s["metric"] == "doc_hit@5"), None),
        "section_hit@5": next((s["mean"] for s in summary if s["category"] == "all" and s["metric"] == "section_hit@5"), None),
        "answer_delivered": next((s["mean"] for s in summary if s["category"] == "all" and s["metric"] == "answer_delivered"), None),
        "tokens_promedio": round(sum(r["tokens_delivered"] for r in ARM_ROWS[arm]) / len(ARM_ROWS[arm]), 1),
        "latencia_s_promedio": round(sum(r["retrieval_s"] for r in ARM_ROWS[arm]) / len(ARM_ROWS[arm]), 3),
    } for arm, summary in ARM_SUMMARIES.items()]))
except ImportError:
    print("Resumen disponible en CSV por brazo.")

bm25 | 23 preguntas | /content/projecto_rag/artifacts/full/results/stage3_comparison_bm25_b4025f7245d2b9e0
dense_bge_m3 | 23 preguntas | /content/projecto_rag/artifacts/full/results/stage3_comparison_dense_bge_m3_fa6ddf5fc51e5245
hybrid_rrf | 23 preguntas | /content/projecto_rag/artifacts/full/results/stage3_comparison_hybrid_rrf_c45c90b96dac551b


,brazo,doc_hit@5,section_hit@5,answer_delivered,tokens_promedio,latencia_s_promedio
0,bm25,0.952381,0.857143,0.952381,2456.8,0.639
1,dense_bge_m3,1.000000,0.904762,0.952381,2461.3,0.667
2,hybrid_rrf,1.000000,0.904762,0.952381,2468.0,0.655


## 13. Ruido estadístico

Se comparan las **mismas preguntas**, con igual presupuesto, con un test de signos exacto bilateral sobre los casos discordantes. También se calcula un intervalo bootstrap de 95% para la diferencia media. La unidad de remuestreo es la pregunta (Etapa 3), no cada chunk. Un cambio de pocos aciertos con 30 preguntas puede deberse al ruido.

In [55]:
PAIRED = []
for prev, cur in [("bm25", "dense_bge_m3"), ("dense_bge_m3", "hybrid_rrf")]:
    for metric in ("doc_hit@5", "section_hit@5", "delivered_section_hit", "answer_delivered"):
        comparison = paired_comparison(ARM_ROWS[prev], ARM_ROWS[cur], metric=metric)
        PAIRED.append({"anterior": prev, "nuevo": cur, **comparison})
comparison_dir = ART / "results" / f"stage3_analysis_{canonical_hash({'spec': RUN_SPEC, 'dense': DENSE_SPEC})[:16]}"
comparison_dir.mkdir(parents=True, exist_ok=True)
write_jsonl(comparison_dir / "paired_comparison.jsonl", PAIRED)
if 'pd' in globals():
    display(pd.DataFrame(PAIRED))
else:
    for c in PAIRED: print(c)

,anterior,nuevo,metric,wins,losses,ties,sign_test_p_two_sided,delta_ci95
0,bm25,dense_bge_m3,doc_hit@5,1,0,20,1.0,"[0.0, 0.14285714285714285]"
1,bm25,dense_bge_m3,section_hit@5,2,1,18,1.0,"[-0.09523809523809523, 0.23809523809523808]"
2,bm25,dense_bge_m3,delivered_section_hit,2,2,17,1.0,"[-0.19047619047619047, 0.19047619047619047]"
3,bm25,dense_bge_m3,answer_delivered,1,1,19,1.0,"[-0.14285714285714285, 0.14285714285714285]"
4,dense_bge_m3,hybrid_rrf,doc_hit@5,0,0,21,1.0,"[0.0, 0.0]"
5,dense_bge_m3,hybrid_rrf,section_hit@5,1,1,19,1.0,"[-0.14285714285714285, 0.14285714285714285]"
6,dense_bge_m3,hybrid_rrf,delivered_section_hit,2,1,18,1.0,"[-0.09523809523809523, 0.19047619047619047]"
7,dense_bge_m3,hybrid_rrf,answer_delivered,1,1,19,1.0,"[-0.14285714285714285, 0.14285714285714285]"


## 14. Diagnóstico cualitativo: casos que cambiaron y rankings por canal

Los indicadores agregados pueden ocultar errores del mismo documento pero artículo incorrecto. Aquí se imprimen los casos discordantes y sus top-5 de BM25, denso y RRF. Se evita afirmar que RRF mejora por su mecanismo sin observar cómo se ordenaron los pasajes.

In [56]:
by_arm = {arm: {r['item_id']: r for r in results} for arm, results in ARM_ROWS.items()}
for before, after in (("bm25", "dense_bge_m3"), ("dense_bge_m3", "hybrid_rrf")):
    print("\nComparación:", before, "->", after)
    discordant = []
    for ident, old in by_arm[before].items():
        new = by_arm[after][ident]
        x, y = old['metrics']['delivered_section_hit'], new['metrics']['delivered_section_hit']
        if x is not None and y is not None and x != y:
            discordant.append(ident)
    print("Casos discordantes:", len(discordant))
    for ident in discordant[:15]:
        item = by_arm[before][ident]
        print("\n", ident, item['question'], "| gold:", item['gold'])
        for arm in ARMS:
            rec = by_arm[arm][ident]
            print(arm, "hit entregado:", rec['metrics']['delivered_section_hit'],
                  "| top-5:", [(c['section_id'], round(c['score'], 4)) for c in rec['candidates'][:5]])

# Verificación de longitud de los pares del reranker (aunque no se active reranking).
for arm, result in ARM_ROWS.items():
    pairs = sum(r['reranker_window_check']['pairs'] for r in result)
    over = sum(r['reranker_window_check']['over_limit'] for r in result)
    print(f"{arm}: exceden 512 tokens {over}/{pairs} pares ({over / pairs:.2%} si hay pares)" if pairs else f"{arm}: sin pares")


Comparación: bm25 -> dense_bge_m3
Casos discordantes: 4

 q004 ¿Las características prometidas en una publicidad obligan al oferente aunque no se repitan en el contrato? | gold: [{'doc_id': 'LEY-24240', 'section_id': 'LEY-24240#ART-8'}]
bm25 hit entregado: 0 | top-5: [('LEY-26994#ANEXO-I/ART-1526', 7.4407), ('LEY-26994#ANEXO-I/ART-1103', 6.5208), ('LEY-26994#ANEXO-I/ART-1252', 6.0204), ('DEC-92-1997#ANEXO-XII/ART-6-H9377efad5af6', 5.9189), ('LEY-26994#ANEXO-I/ART-1803', 5.7864)]
dense_bge_m3 hit entregado: 1 | top-5: [('LEY-26994#ANEXO-I/ART-1803', 0.6418), ('LEY-24240#ART-8', 0.6349), ('LEY-26994#ANEXO-I/ART-1103', 0.618), ('LEY-26994#ANEXO-I/ART-1807', 0.6147), ('LEY-24240#ART-7', 0.5989)]
hybrid_rrf hit entregado: 1 | top-5: [('LEY-26994#ANEXO-I/ART-1103', 0.032), ('LEY-26994#ANEXO-I/ART-1803', 0.0318), ('LEY-24240#ART-8', 0.0295), ('DEC-92-1997#ANEXO-XII/ART-6-H9377efad5af6', 0.0284), ('LEY-26994#ANEXO-I/ART-1612', 0.0261)]

 q015 Según el texto de la Ley 25.649 de nuestra base, ¿

## 15. Selección reproducible de la configuración para las Etapas 4 y 5

**Decisión metodológica:** la selección automática se hace **solo sobre `dev`**, maximizando la proporción de secciones gold entregadas y desempantando por menor latencia. Es una regla simple y explícita; antes del informe final conviene revisar los casos discordantes y registrar la decisión humana en `decision_log.md`. En `test`, se exige cargar la decisión ya guardada de `dev`, sin volver a elegir.

Se exporta un manifiesto y los contextos por pregunta. Para consultas nuevas de la Etapa 5 es necesario reconstruir el recuperador (BM25/embeddings/encoder) desde el corpus y el manifiesto; no basta con los contextos pregenerados de Etapa 4.

In [57]:
SELECTION_FILE = ART / 'index' / 'stage3_selected_retriever.json'
if EVAL_SPLIT == 'dev':
    def select_score(arm):
        vals = [r['metrics']['delivered_section_hit'] for r in ARM_ROWS[arm]
                if r['metrics']['delivered_section_hit'] is not None]
        hit = sum(vals) / len(vals) if vals else float('-inf')
        latency = sum(r['retrieval_s'] for r in ARM_ROWS[arm]) / len(ARM_ROWS[arm])
        return (hit, -latency)

    # Regla fijada en dev:
    # maximizar entrega de sección gold y, ante empate,
    # preferir la configuración más simple.
    prioridad = ["bm25", "dense_bge_m3", "hybrid_rrf"]

    mejor_hit = max(select_score(arm)[0] for arm in ARMS)

    SELECTED_ARM = next(
        arm for arm in prioridad
        if select_score(arm)[0] == mejor_hit
    )


    CHOICE = {"selected_arm": SELECTED_ARM, "selected_on": "dev", "corpus_hash": corpus_hash,
              "frozen_eval_hash": canonical_hash(FROZEN), "index_dir": str(INDEX_DIR),
              "dense_dir": str(DENSE_DIR), "dense_model": DENSE_MODEL_ID,
              "dense_revision": DENSE_REVISION, "dense_spec": DENSE_SPEC,
              "budget_tokens": CFG.budget_tokens, "candidate_k": CFG.candidate_k,
              "rrf_k": 60, "seed": SEED,
              "selection_rule": (
                  "max delivered_section_hit on dev; "
                  "tie break simpler retrieval architecture"
              )
    }
    write_json(SELECTION_FILE, CHOICE)
else:
    if not SELECTION_FILE.exists():
        raise RuntimeError('Primero corré dev y fijá la selección; test no puede elegir un nuevo brazo.')
    CHOICE = json.loads(SELECTION_FILE.read_text(encoding='utf-8'))
    if CHOICE['corpus_hash'] != corpus_hash or CHOICE['frozen_eval_hash'] != canonical_hash(FROZEN):
        raise RuntimeError('El brazo seleccionado corresponde a otro corpus o benchmark.')
    SELECTED_ARM = CHOICE['selected_arm']
SELECTED_RETRIEVER = ARMS[SELECTED_ARM]
print('Brazo seleccionado:', SELECTED_ARM)
print('Manifiesto:', SELECTION_FILE)

# Contrato que reutilizan las Etapas 4 y 5 en esta sesión:
# SELECTED_RETRIEVER.retrieve(query) -> chunks ordenados con doc_id, section_id, text y score.
def retrieve_stage3(query, budget_tokens=None):
    ranked = SELECTED_RETRIEVER.retrieve(query)
    budget = CFG.budget_tokens if budget_tokens is None else budget_tokens
    delivered, skipped, used = deliver_by_budget(ranked, GEN_TOKENIZER, budget)
    return {"query": query, "arm": SELECTED_ARM, "ranked": ranked,
            "passages": delivered, "skipped_chunk_ids": skipped,
            "tokens_delivered": used, "budget_tokens": budget,
            "context": render_context(delivered)}

check = retrieve_stage3(items[0]['question'])
assert check['tokens_delivered'] <= CFG.budget_tokens
assert all({'doc_id','section_id','chunk_id','text','score'} <= set(p) for p in check['passages'])
print('Interfaz de Etapas 4/5 verificada; pasajes:', len(check['passages']))

Brazo seleccionado: bm25
Manifiesto: /content/projecto_rag/artifacts/full/index/stage3_selected_retriever.json
Interfaz de Etapas 4/5 verificada; pasajes: 11


## 16. Límites del experimento y lectura para el informe

1. **No se atribuye causalidad al prefijo:** BGE-M3 usa un prefijo contextual distinto del canal lexical; comparar ambos muestra el comportamiento de dos sistemas configurados, no un efecto aislado del modelo de embeddings.
2. **No se prueba reranking ni entrega de padres:** se omiten para priorizar la Etapa 5. La ventana del reranker se audita sin cargar sus pesos.
3. **Las citas y respuestas fundamentadas aún no están evaluadas:** `answer_delivered` solo comprueba presencia lexical. La Etapa 4 verificará las respuestas del generador.
4. **El presupuesto de 2.500 tokens es del bloque de evidencia:** la Etapa 4 debe contar además sistema, historial, marcadores y reserva de salida bajo `num_ctx` explícito.
5. **La selección en `dev` no constituye resultado final.** Una vez revisada y registrada, correr `test` sin retocar el pipeline.

**Para el informe:** describir por qué se probó cada brazo, las métricas por categoría, las cotas de ruido, los casos discordantes, el costo de embeddings y qué componentes se descartaron para priorizar las Etapas 4 y 5.

## 17. Exportar el recuperador para notebooks 04 y 05

Se escribe `stage3_runtime.py` en el directorio del proyecto: encapsula la carga del índice BM25, la matriz densa, la revisión del encoder y la configuración seleccionada. En las siguientes etapas se usa `from stage3_runtime import load_stage3` y `load_stage3(environment.artifacts_dir(False))`. Esta carga reconstruye BM25 y lee embeddings guardados; no requiere volver a ejecutar las evaluaciones.


In [58]:
from pathlib import Path
RUNTIME_SOURCE = '"""Etapa 3 inicial: BM25, chunks estructurales y entrega por tokens.\n\nEl notebook incluye una copia de este módulo para poder abrirse solo en Colab.\nNo llama al LLM ni modifica el conjunto de evaluación congelado.\n"""\nfrom __future__ import annotations\n\nfrom dataclasses import asdict, dataclass\nfrom datetime import datetime, timezone\nfrom pathlib import Path\nimport hashlib\nimport importlib.metadata\nimport json\nimport math\nimport random\nimport re\nimport time\nimport unicodedata\n\n\nSTOP = set(\n    "de la que el en y los se del las un por con no una su para "\n    "es al lo como más a o sus le les".split()\n)\nEVAL_FILES = ("single_turn.jsonl", "dialogues.jsonl", "scenarios.jsonl")\n\n\ndef canonical_hash(value):\n    blob = json.dumps(value, sort_keys=True, ensure_ascii=False, separators=(",", ":"))\n    return hashlib.sha256(blob.encode("utf-8")).hexdigest()\n\n\ndef file_hash(path):\n    return hashlib.sha256(Path(path).read_bytes()).hexdigest()\n\n\ndef read_jsonl(path):\n    return [json.loads(line) for line in Path(path).read_text(encoding="utf-8").splitlines() if line.strip()]\n\n\ndef write_json(path, value):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    path.write_text(json.dumps(value, ensure_ascii=False, indent=2) + "\\n", encoding="utf-8")\n\n\ndef write_jsonl(path, rows):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    path.write_text("".join(json.dumps(r, ensure_ascii=False) + "\\n" for r in rows), encoding="utf-8")\n\n\ndef verify_frozen_files(eval_dir):\n    """Verifica los tres conjuntos, sin escribir ni volver a congelar."""\n    eval_dir = Path(eval_dir)\n    frozen = json.loads((eval_dir / "FROZEN.json").read_text(encoding="utf-8"))\n    for name in EVAL_FILES:\n        rows = read_jsonl(eval_dir / name)\n        spec = frozen["files"][name]\n        if len(rows) != spec["rows"] or canonical_hash(rows) != spec["sha256_canonical"]:\n            raise ValueError(f"Evaluación modificada: {name}. No se recalculó FROZEN.json.")\n    return frozen\n\n\n@dataclass(frozen=True)\nclass RetrievalConfig:\n    name: str = "bm25"\n    child_tokens: int = 320\n    overlap_fraction: float = 0.15\n    budget_tokens: int = 2500\n    candidate_k: int = 100\n    metric_ks: tuple = (1, 3, 5)\n    bm25_k1: float = 1.5\n    bm25_b: float = 0.75\n    seed: int = 1337\n    bootstrap_samples: int = 2000\n\n    def __post_init__(self):\n        if self.name not in ("bm25", "dense_bge_m3", "hybrid_rrf"):\n            raise ValueError("Brazo no reconocido.")\n        if self.child_tokens < 32 or not 0 <= self.overlap_fraction < 0.5:\n            raise ValueError("child_tokens >= 32 y overlap_fraction en [0, 0.5).")\n        if self.budget_tokens < 1 or self.candidate_k < 1:\n            raise ValueError("Presupuesto y candidatos deben ser positivos.")\n        if not self.metric_ks or min(self.metric_ks) < 1 or max(self.metric_ks) > self.candidate_k:\n            raise ValueError("metric_ks debe estar entre 1 y candidate_k.")\n        if self.bootstrap_samples < 1 or self.bm25_k1 <= 0 or not 0 <= self.bm25_b <= 1:\n            raise ValueError("Parámetros BM25 o bootstrap inválidos.")\n\n\ndef token_count(tokenizer, text):\n    return len(tokenizer.encode(text, add_special_tokens=False))\n\n\ndef chunk_documents(documents, tokenizer, cfg):\n    """Offsets del texto original; nunca mezcla secciones ni decodifica tokens cortados.\n\n    Dimensiona en el tokenizer del reranker, prefijo incluido. La entrega se cuenta\n    aparte en el tokenizer de Qwen. No carga los pesos de ningún modelo.\n    """\n    chunks = []\n    doc_ids = set()\n    for doc in sorted(documents, key=lambda d: d["doc_id"]):\n        doc_id = doc["doc_id"]\n        if doc_id in doc_ids:\n            raise ValueError(f"doc_id duplicado: {doc_id}")\n        doc_ids.add(doc_id)\n        metadata = doc.get("metadata", {})\n        seen_sections = set()\n        for sec in sorted(doc["sections"], key=lambda s: s["start"]):\n            sec_id = sec["section_id"]\n            if sec_id in seen_sections:\n                raise ValueError(f"section_id duplicado: {sec_id}")\n            seen_sections.add(sec_id)\n            if not 0 <= sec["start"] < sec["end"] <= len(doc["text"]):\n                raise ValueError(f"Offsets inválidos: {sec_id}")\n            body = doc["text"][sec["start"]:sec["end"]]\n            if not body.strip():\n                continue\n            # Prefijo mínimo fijo: identificador normativo y título de la sección.\n            # No recorta títulos o texto jurídico en silencio.\n            prefix = f"{doc_id} | {sec[\'title\']}\\n"\n            prefix_n = token_count(tokenizer, prefix)\n            room = cfg.child_tokens - prefix_n\n            if room < 16:\n                raise ValueError(f"Prefijo demasiado largo para el chunk: {sec_id}")\n            encoded = tokenizer(body, add_special_tokens=False, return_offsets_mapping=True, truncation=False)\n            offsets = encoded["offset_mapping"]\n            if not offsets:\n                raise ValueError(f"Texto no vacío sin tokens: {sec_id}")\n            start_i = 0\n            while start_i < len(offsets):\n                end_i = min(start_i + room, len(offsets))\n                start_char = 0 if start_i == 0 else offsets[start_i][0]\n                end_char = len(body) if end_i == len(offsets) else offsets[end_i][0]\n                text = body[start_char:end_char]\n                # Retokenizar el fragmento aislado puede cambiar sus bordes BPE.\n                while token_count(tokenizer, prefix + text) > cfg.child_tokens:\n                    end_i -= 1\n                    if end_i <= start_i:\n                        raise ValueError(f"No se pudo dimensionar un fragmento: {sec_id}")\n                    end_char = offsets[end_i][0]\n                    text = body[start_char:end_char]\n                if end_char <= start_char:\n                    raise ValueError(f"Fragmento sin avance: {sec_id}")\n                if text.strip():\n                    absolute_start, absolute_end = sec["start"] + start_char, sec["start"] + end_char\n                    identity = {"section_id": sec_id, "start": absolute_start, "end": absolute_end, "text": text}\n                    chunks.append({\n                        "chunk_id": f"{sec_id}@{absolute_start}:{absolute_end}-{canonical_hash(identity)[:12]}",\n                        "doc_id": doc_id, "section_id": sec_id,\n                        "title": doc["title"], "section_title": sec["title"],\n                        "start": absolute_start, "end": absolute_end,\n                        "text": text, "prefix": prefix, "index_text": prefix + text,\n                        "reranker_tokens": token_count(tokenizer, prefix + text),\n                        "metadata": metadata,\n                    })\n                if end_i == len(offsets):\n                    break\n                overlap = int((end_i - start_i) * cfg.overlap_fraction)\n                start_i = max(start_i + 1, end_i - overlap)\n    if not chunks:\n        raise ValueError("El corpus no produjo chunks.")\n    if len({c["chunk_id"] for c in chunks}) != len(chunks):\n        raise ValueError("IDs de chunk duplicados.")\n    return chunks\n\n\ndef validate_gold_in_index(items, chunks):\n    available = {(c["doc_id"], c["section_id"]) for c in chunks}\n    missing = []\n    for item in items:\n        if item["answerable"] and not item["gold"]:\n            raise ValueError(f"Ítem respondible sin gold: {item[\'item_id\']}")\n        for gold in item["gold"]:\n            pair = (gold["doc_id"], gold["section_id"])\n            if pair not in available:\n                missing.append((item["item_id"], pair))\n    if missing:\n        raise ValueError(f"Gold ausente del índice: {missing[:10]}")\n\n\nclass BM25Retriever:\n    def __init__(self, chunks, cfg):\n        import bm25s\n        import Stemmer\n        self.bm25s = bm25s\n        self.stemmer = Stemmer.Stemmer("spanish")\n        self.cfg = cfg\n        self.chunks = chunks\n        texts = [" ".join(self.words(c["index_text"])) for c in chunks]\n        tokens = self.tokenize(texts)\n        self.index = bm25s.BM25(k1=cfg.bm25_k1, b=cfg.bm25_b, method="lucene")\n        self.index.index(tokens, show_progress=False)\n\n    def words(self, text):\n        text = unicodedata.normalize("NFC", text.lower())\n        text = re.sub(r"\\b(\\d{2})\\.(\\d{3})\\b", r"\\1\\2", text)\n        return [self.stemmer.stemWord(w) for w in re.findall(r"\\w+", text) if w not in STOP]\n\n    def tokenize(self, texts):\n        return self.bm25s.tokenize(texts, stopwords=[], token_pattern=r"(?u)\\b\\w+\\b", show_progress=False)\n\n    def retrieve(self, query):\n        words = self.words(query)\n        if not words:\n            return []\n        indices, scores = self.index.retrieve(\n            self.tokenize([" ".join(words)]),\n            k=min(self.cfg.candidate_k, len(self.chunks)), show_progress=False,\n        )\n        ranked = [\n            {**self.chunks[int(i)], "score": float(score)}\n            for i, score in zip(indices[0], scores[0]) if float(score) > 0\n        ]\n        return sorted(ranked, key=lambda c: (-c["score"], c["chunk_id"]))\n\n\ndef passage_block(chunk):\n    # Este mismo bloque se entrega a la etapa 4, incluidos sus identificadores.\n    return f"[{chunk[\'doc_id\']} · {chunk[\'section_id\']} · {chunk[\'chunk_id\']}]\\n{chunk[\'index_text\']}"\n\n\ndef render_context(chunks):\n    return "\\n\\n".join(passage_block(c) for c in chunks)\n\n\ndef deliver_by_budget(ranked, tokenizer, budget):\n    delivered, skipped = [], []\n    for c in ranked:\n        if token_count(tokenizer, render_context(delivered + [c])) <= budget:\n            delivered.append(c)\n        else:\n            skipped.append(c["chunk_id"])\n    used = token_count(tokenizer, render_context(delivered))\n    if used > budget:\n        raise AssertionError("La entrega excedió el presupuesto.")\n    return delivered, skipped, used\n\n\ndef normalize_answer(text):\n    text = unicodedata.normalize("NFD", text.lower())\n    text = "".join(c for c in text if unicodedata.category(c) != "Mn")\n    text = re.sub(r"\\(\\s*\\d+\\s*\\)", " ", text)\n    text = re.sub(r"(?<=\\d)[.,\\s](?=\\d{3}(?:\\D|$))", "", text)\n    text = re.sub(r"[^\\w\\s]", " ", text)\n    return " ".join(text.split())\n\n\ndef contains_key(text, keys):\n    normalized = f" {normalize_answer(text)} "\n    return any(\n        bool(normalize_answer(key)) and f" {normalize_answer(key)} " in normalized\n        for key in keys or []\n    )\n\n\ndef score_retrieval(item, ranked, delivered, ks):\n    """k = posiciones de chunks recuperados; MRR truncado a candidate_k.\n\n    Fuera del corpus: métricas de recuperación nulas, nunca fallos artificiales.\n    answer_delivered busca claves SOLO en cuerpos, no en títulos/IDs del prefijo.\n    """\n    metrics = {"max_score": ranked[0]["score"] if ranked else 0.0}\n    gold = {(g["doc_id"], g["section_id"]) for g in item["gold"]}\n    gold_docs = {g[0] for g in gold}\n    answerable = item["answerable"]\n    for k in ks:\n        top = ranked[:k]\n        metrics[f"doc_hit@{k}"] = int(any(c["doc_id"] in gold_docs for c in top)) if answerable else None\n        metrics[f"section_hit@{k}"] = int(any((c["doc_id"], c["section_id"]) in gold for c in top)) if answerable else None\n    for level in ["doc", "section"]:\n        position = next((i for i, c in enumerate(ranked, 1) if (\n            c["doc_id"] in gold_docs if level == "doc" else (c["doc_id"], c["section_id"]) in gold\n        )), None)\n        metrics[f"{level}_mrr_candidates"] = (1.0 / position if position else 0.0) if answerable else None\n    metrics["delivered_section_hit"] = int(any((c["doc_id"], c["section_id"]) in gold for c in delivered)) if answerable else None\n    keys = item.get("answer_key")\n    metrics["answer_delivered"] = int(any(contains_key(c["text"], keys) for c in delivered)) if answerable and keys else None\n    return metrics\n\n\ndef reranker_overflow(query, candidates, tokenizer, max_length=512):\n    lengths = [len(tokenizer(query, c["index_text"], add_special_tokens=True, truncation=False)["input_ids"]) for c in candidates]\n    return {"pairs": len(lengths), "over_limit": sum(n > max_length for n in lengths), "max_pair_tokens": max(lengths, default=0)}\n\n\ndef evaluate_items(items, retriever, generator_tokenizer, reranker_tokenizer, run_spec):\n    run_key = canonical_hash(run_spec)\n    rows = []\n    cfg = retriever.cfg\n    for item in items:\n        t0 = time.perf_counter()\n        ranked = retriever.retrieve(item["question"])\n        search_s = time.perf_counter() - t0\n        t1 = time.perf_counter()\n        delivered, skipped, used = deliver_by_budget(ranked, generator_tokenizer, cfg.budget_tokens)\n        packing_s = time.perf_counter() - t1\n        rows.append({\n            "run_key": run_key, "run_spec": run_spec,\n            "item_id": item["item_id"], "question": item["question"],\n            "category": item["category"], "split": item["split"],\n            "answerable": item["answerable"], "gold": item["gold"],\n            "answer_key": item.get("answer_key"), "expected_verdict": item.get("expected_verdict"),\n            "config": asdict(cfg),\n            "metrics": score_retrieval(item, ranked, delivered, cfg.metric_ks),\n            "tokens_delivered": used, "budget_tokens": cfg.budget_tokens,\n            "search_s": search_s, "packing_s": packing_s, "retrieval_s": search_s + packing_s,\n            "reranker_window_check": reranker_overflow(item["question"], ranked, reranker_tokenizer),\n            "candidates": [{k: c[k] for k in ["chunk_id", "doc_id", "section_id", "score"]} for c in ranked],\n            "delivered": [{**c, "passage": passage_block(c)} for c in delivered],\n            "skipped_chunk_ids": skipped,\n        })\n    return rows\n\n\ndef bootstrap_mean(values, seed=1337, samples=2000):\n    if not values:\n        return None, None\n    rng = random.Random(seed)\n    means = sorted(sum(rng.choices(values, k=len(values))) / len(values) for _ in range(samples))\n    return means[int(0.025 * (samples - 1))], means[int(0.975 * (samples - 1))]\n\n\ndef summary_rows(rows, cfg):\n    result = []\n    for category in ["all"] + sorted({r["category"] for r in rows}):\n        group = [r for r in rows if category == "all" or r["category"] == category]\n        if not group:\n            continue\n        for metric in rows[0]["metrics"]:\n            if metric == "max_score":\n                continue  # Los puntajes OOC se inspeccionan aparte; no son exactitud.\n            values = [r["metrics"][metric] for r in group if r["metrics"][metric] is not None]\n            low, high = bootstrap_mean(values, cfg.seed, cfg.bootstrap_samples)\n            result.append({"category": category, "metric": metric, "n": len(values),\n                           "mean": sum(values) / len(values) if values else None,\n                           "ci95_low": low, "ci95_high": high})\n    return result\n\n\ndef paired_comparison(previous, current, metric="answer_delivered"):\n    """Para futuras filas: mismos ítems, split, benchmark y presupuesto."""\n    a, b = {r["item_id"]: r for r in previous}, {r["item_id"]: r for r in current}\n    if len(a) != len(previous) or len(b) != len(current) or a.keys() != b.keys():\n        raise ValueError("La comparación requiere los mismos ítems, sin duplicados.")\n    wins = losses = ties = 0\n    differences = []\n    for ident in sorted(a):\n        x, y = a[ident], b[ident]\n        if (x["split"] != y["split"] or x["budget_tokens"] != y["budget_tokens"]\n                or x["run_spec"]["eval"] != y["run_spec"]["eval"]\n                or x["run_spec"]["corpus_hash"] != y["run_spec"]["corpus_hash"]):\n            raise ValueError("Evaluación, corpus, split o presupuesto diferentes.")\n        p, q = x["metrics"][metric], y["metrics"][metric]\n        if p is None or q is None:\n            continue\n        if p not in (0, 1) or q not in (0, 1):\n            raise ValueError("El test de signos requiere una métrica binaria.")\n        wins += q > p\n        losses += q < p\n        ties += q == p\n        differences.append(q - p)\n    n = wins + losses\n    p_value = min(1.0, 2 * sum(math.comb(n, i) for i in range(min(wins, losses) + 1)) / 2**n) if n else 1.0\n    low, high = bootstrap_mean(differences)\n    return {"metric": metric, "wins": wins, "losses": losses, "ties": ties,\n            "sign_test_p_two_sided": p_value, "delta_ci95": [low, high]}\n\n\ndef package_versions():\n    versions = {}\n    for name in ["bm25s", "PyStemmer", "transformers", "tokenizers", "numpy", "scipy", "huggingface-hub"]:\n        try:\n            versions[name] = importlib.metadata.version(name)\n        except importlib.metadata.PackageNotFoundError:\n            versions[name] = None\n    return versions\n\n\ndef project_sources():\n    """Huellas de los auxiliares cargados, además del código de esta etapa."""\n    import sys\n    sources = {}\n    for name in ["environment", "schema", "console", "store", "llm"]:\n        module = sys.modules.get(name)\n        path = getattr(module, "__file__", None)\n        if path and Path(path).is_file():\n            sources[name] = {"sha256": file_hash(path)}\n    return sources\n\n\ndef save_index(retriever, index_dir, manifest):\n    index_dir = Path(index_dir)\n    index_dir.mkdir(parents=True, exist_ok=True)\n    existing = index_dir / "manifest.json"\n    if existing.exists() and canonical_hash(json.loads(existing.read_text(encoding="utf-8"))) != canonical_hash(manifest):\n        raise ValueError("El directorio de índice pertenece a otra configuración.")\n    retriever.index.save(str(index_dir / "bm25"))\n    write_jsonl(index_dir / "chunks.jsonl", retriever.chunks)\n    write_json(existing, manifest)\n\n\ndef utc_now():\n    return datetime.now(timezone.utc).isoformat()\n\n# -----------------------------------------------------------------------------\n# Public interface for Stage 4 (grounded answers) and Stage 5 (conversations).\n# Rebuilds BM25 and opens the cached dense matrix. Never reads test outcomes.\n# -----------------------------------------------------------------------------\n\nclass Stage3Runtime:\n    def __init__(self, artifacts_dir, device=None):\n        from transformers import AutoTokenizer\n        from sentence_transformers import SentenceTransformer\n        import numpy as np\n        import torch\n\n        art = Path(artifacts_dir)\n        choice_file = art / \'index\' / \'stage3_selected_retriever.json\'\n        if not choice_file.exists():\n            raise FileNotFoundError(f\'Run Stage 3 on dev first: {choice_file}\')\n        self.choice = json.loads(choice_file.read_text(encoding=\'utf-8\'))\n        self.name = self.choice[\'selected_arm\']\n        index_dir = Path(self.choice[\'index_dir\'])\n        dense_dir = Path(self.choice[\'dense_dir\'])\n        if not index_dir.exists():\n            raise FileNotFoundError(f\'Missing saved stage3 index: {index_dir}\')\n        manifest = json.loads((index_dir / \'manifest.json\').read_text(encoding=\'utf-8\'))\n        if manifest[\'corpus_hash\'] != self.choice[\'corpus_hash\']:\n            raise ValueError(\'Selected config and index have different corpus hashes.\')\n        self.chunks = read_jsonl(index_dir / \'chunks.jsonl\')\n        if canonical_hash(self.chunks) != manifest[\'chunks_hash\']:\n            raise ValueError(\'Stage3 chunks do not match their saved manifest.\')\n        self.cfg = RetrievalConfig(**manifest[\'config\'])\n        self.tokenizer = AutoTokenizer.from_pretrained(\n            manifest[\'tokenizers\'][\'generator\'][\'model\'],\n            revision=manifest[\'tokenizers\'][\'generator\'][\'revision\'])\n        self.lexical = BM25Retriever(self.chunks, self.cfg) if self.name in (\'bm25\', \'hybrid_rrf\') else None\n        self.encoder = None\n        self.matrix = None\n        if self.name in (\'dense_bge_m3\', \'hybrid_rrf\'):\n            if json.loads((dense_dir / \'manifest.json\').read_text(encoding=\'utf-8\')) != self.choice[\'dense_spec\']:\n                raise ValueError(\'Dense cache manifest changed.\')\n            self.matrix = np.load(dense_dir / \'matrix.npy\', allow_pickle=False)\n            if self.matrix.shape[0] != len(self.chunks):\n                raise ValueError(\'Dense matrix and saved chunks differ.\')\n            target_device = device or (\'cuda\' if torch.cuda.is_available() else \'cpu\')\n            self.encoder = SentenceTransformer(\n                self.choice[\'dense_model\'], revision=self.choice[\'dense_revision\'],\n                device=target_device, trust_remote_code=False)\n            self.encoder.max_seq_length = self.choice[\'dense_spec\'][\'max_seq_length\']\n\n    def retrieve_ranked(self, query):\n        import numpy as np\n        if not isinstance(query, str) or not query.strip():\n            raise ValueError(\'Query must be a nonempty string.\')\n        lexical_ranked = self.lexical.retrieve(query) if self.lexical is not None else None\n        dense_ranked = None\n        if self.encoder is not None:\n            vector = self.encoder.encode([query], normalize_embeddings=True,\n                                         convert_to_numpy=True)[0].astype(np.float32)\n            similarities = self.matrix @ vector\n            k = min(self.cfg.candidate_k, len(self.chunks))\n            ids = np.argpartition(-similarities, k - 1)[:k]\n            ids = sorted(ids, key=lambda i: (-float(similarities[i]), self.chunks[i][\'chunk_id\']))\n            dense_ranked = [{**self.chunks[int(i)], \'score\': float(similarities[i])} for i in ids]\n        if self.name == \'bm25\':\n            return lexical_ranked\n        if self.name == \'dense_bge_m3\':\n            return dense_ranked\n        if self.name == \'hybrid_rrf\':\n            scores, by_id = {}, {}\n            for channel in (lexical_ranked, dense_ranked):\n                for rank, c in enumerate(channel, start=1):\n                    key = c[\'chunk_id\']\n                    scores[key] = scores.get(key, 0.0) + 1.0 / (self.choice[\'rrf_k\'] + rank)\n                    by_id[key] = c\n            best = sorted(scores, key=lambda key: (-scores[key], key))[:self.cfg.candidate_k]\n            return [{**by_id[key], \'score\': scores[key]} for key in best]\n        raise ValueError(\'Unknown selected arm: \' + self.name)\n\n    def retrieve(self, query, budget_tokens=None):\n        budget = self.cfg.budget_tokens if budget_tokens is None else int(budget_tokens)\n        if budget <= 0:\n            raise ValueError(\'Budget must be positive.\')\n        ranked = self.retrieve_ranked(query)\n        delivered, skipped, used = deliver_by_budget(ranked, self.tokenizer, budget)\n        return {\'query\': query, \'arm\': self.name, \'ranked\': ranked,\n                \'passages\': delivered, \'skipped_chunk_ids\': skipped,\n                \'tokens_delivered\': used, \'budget_tokens\': budget,\n                \'context\': render_context(delivered)}\n\n\ndef load_stage3(artifacts_dir, device=None):\n    """For 04 and 05: retrieval = load_stage3(environment.artifacts_dir(False))."""\n    return Stage3Runtime(artifacts_dir, device=device)\n'
Path("stage3_runtime.py").write_text(RUNTIME_SOURCE, encoding="utf-8")
print("Módulo generado:", Path("stage3_runtime.py").resolve())


Módulo generado: /content/projecto_rag/stage3_runtime.py



### Validación posterior del conjunto de evaluación

El conjunto de evaluación fue congelado antes de completar su revisión humana documentada.

Posteriormente se revisaron los 30 ítems de `single_turn.jsonl` (7 de desarrollo y 23 de prueba), contrastando las preguntas, las respuestas esperadas y las referencias gold con el corpus jurídico.

Los 30 ítems fueron aprobados por una integrante del grupo. No se identificaron errores de contenido que requirieran modificar el benchmark.

La advertencia sobre revisión pendiente corresponde a las notas originales del archivo congelado. La revisión posterior se documentó de manera independiente en:

- `acta_revision_humana_single_turn_2026-10-29.md`
- `revision_humana_single_turn_2026-10-29.csv`

La validación se realizó con asistencia de IA para organizar y contrastar evidencias, y confirmación final humana.

No se modificaron `single_turn.jsonl` ni `FROZEN.json`.
